# Why doesn't retrieval help?

Per-instance comparison of the reference config (`current`) vs. `+ retrieval`, same judge, same instances.

In [ ]:
import html
import json
import re
import sys
from functools import cache
from pathlib import Path

import pandas as pd
import yaml
from IPython.display import HTML, display
from scipy.stats import binomtest
from sklearn.metrics import f1_score

ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "src").is_dir() and (p / "output").is_dir()
)
sys.path.insert(0, str(ROOT / "src"))

from novelty_eval.analysis.artifacts import (
    _extract_exclude_set_from_filtered_report, _extract_instances_path)

# The merges behind the paper's controlled-study figure.
MERGES = {"human-only": "merge_hvh_all_metrics", "human+gen": "merge_vanilla_all_metrics"}
MODELS = ["claude-sonnet-4-5", "claude-opus-4-5", "claude-opus-4-6", "gpt-5.1", "gpt-5.2", "gpt-5.4"]

# Pairwise per-instance score: "strict" (tie = wrong, the paper's main figure) or "soft" (tie = 0.5).
PAIRWISE_METRIC = "strict"

## 0. Load

One row per (track, mode, model, instance). Blocklisted instances are dropped and missing predictions count as wrong, as in the paper.

In [ ]:
# <mode>-<track>_<ablation>-<model>_<timestamp>
NAME_RE = re.compile(
    r"^(pairwise|pointwise)-[^_]+_(current|retrieval)-(.+)_\d{4}-\d{2}-\d{2}_\d{2}-\d{2}-\d{2}$")

DIRS = {}  # (track, mode, model, cond) -> artifact dir
for track, merge in MERGES.items():
    for d in sorted((ROOT / "output/ablation_sweeps" / merge / "artifacts").iterdir()):
        if m := NAME_RE.match(d.name):
            key = (track, m[1], m[3], m[2])
            assert key not in DIRS, f"two runs for {key}"
            DIRS[key] = d


def load_run(d, mode):
    """gold/pred: pointwise 1 = novel; pairwise = position (0/1) of the idea in the comparison, pred 2 = tie."""
    instances = yaml.safe_load((ROOT / _extract_instances_path(d)).read_text())
    scores = json.loads(next(d.glob("run_*/scores.json")).read_text())
    excluded = _extract_exclude_set_from_filtered_report(d)
    rows = []
    for iid, inst in instances.items():
        iid = str(iid)
        if iid in excluded:
            continue
        s = scores.get(iid)
        if mode == "pointwise":
            gold = int(inst["label"] == "POSITIVE")
            pred = None if s is None else int(s["prediction"])
        else:
            comp = s["comparisons"][0] if s else None
            winners = {str(w) for w in inst["expected_winners"]}
            gold = None if comp is None else int(str(comp["idea_0"]) not in winners)
            pred = None if comp is None else comp["winner"]
        rows.append({"id": iid, "gold": gold, "pred": pred, "ok": pred is not None and pred == gold, "tie": pred == 2})
    return pd.DataFrame(rows).set_index("id")


def macro_f1(gold, pred):
    """Missing predictions count as wrong, as in the paper."""
    pred = pd.to_numeric(pred).fillna(1 - gold)
    return f1_score(gold.astype(int), pred.astype(int), average="macro")


RUNS = {k: load_run(d, k[1]) for k, d in DIRS.items()}

# Must reproduce the filtered reports the paper's figures are built from.
CHECKS = {
    "pointwise": [(r"Mean F1 \(macro\):", lambda r: macro_f1(r.gold, r.pred))],
    "pairwise": [(r"Mean LLM Pairwise Accuracy \(strict\):", lambda r: r.ok.mean()),
                 (r"Mean LLM Pairwise Accuracy \(with ties\):", lambda r: (r.ok + 0.5 * r.tie).mean())],
}
for key, r in RUNS.items():
    report = (DIRS[key] / "filtered_accuracy_report.txt").read_text()
    for pat, metric in CHECKS[key[1]]:
        reported = float(re.search(pat + r"\s+([\d.]+)", report)[1])
        assert abs(metric(r) - reported) < 1e-3, (key, pat, metric(r), reported)
print(f"{len(RUNS)} runs, all match their filtered reports")

In [ ]:
# Paired view: one row per instance, reference vs. retrieval side by side.
frames = []
for track, mode, model in sorted({k[:3] for k in RUNS}):
    ref, ret = RUNS[track, mode, model, "current"], RUNS[track, mode, model, "retrieval"]
    assert ref.index.equals(ret.index), (track, mode, model)
    frames.append(pd.DataFrame({
        "track": track, "mode": mode, "model": model,
        "gold": ref.gold.fillna(ret.gold),
        "ref_pred": ref.pred, "ret_pred": ret.pred,
        "ref_ok": ref.ok, "ret_ok": ret.ok,
        "ref_tie": ref.tie, "ret_tie": ret.tie,
    }))
PAIRED = pd.concat(frames).reset_index()

# Outcome ref → retrieval, e.g. "tie→✓" (pointwise has no ties).
state = lambda ok, tie: pd.Series("✗", ok.index).mask(ok, "✓").mask(tie, "tie")
PAIRED["transition"] = state(PAIRED.ref_ok, PAIRED.ref_tie) + "→" + state(PAIRED.ret_ok, PAIRED.ret_tie)
PAIRED.groupby(["track", "mode"]).size().unstack()

## 1. Helped vs. hurt

helped / hurt = the per-instance score went up / down with retrieval. `p`: sign test on helped vs. hurt (= exact McNemar for 0/1 scores).
Pointwise: macro-F1, flips = instance correctness. Pairwise score is set by `PAIRWISE_METRIC`.

In [ ]:
# Score-changing pairwise transitions to break the flips into, per metric.
TRANSITIONS = {
    "strict": ["✗→✓", "tie→✓", "✓→✗", "✓→tie"],
    "soft": ["✗→✓", "tie→✓", "✗→tie", "✓→✗", "✓→tie", "tie→✗"],
}
PAIRWISE_NAMES = {
    "strict": "strict accuracy (tie = wrong)",
    "soft": "soft accuracy (tie = 0.5)",
}


def flip_stats(g):
    pointwise = g["mode"].iat[0] == "pointwise"
    credit = 0.5 if PAIRWISE_METRIC == "soft" else 0.0  # pointwise has no ties
    ref, ret = g.ref_ok + credit * g.ref_tie, g.ret_ok + credit * g.ret_tie
    helped, hurt = ret > ref, ret < ref
    n_flips = helped.sum() + hurt.sum()

    if pointwise:
        f1_ref, f1_ret = macro_f1(g.gold, g.ref_pred), macro_f1(g.gold, g.ret_pred)
        row = {"n": len(g), "F1_ref": 100 * f1_ref, "ΔF1": 100 * (f1_ret - f1_ref)}
    else:
        name = PAIRWISE_METRIC
        row = {"n": len(g), f"{name}_ref": 100 * ref.mean(), f"Δ{name}": 100 * (ret.mean() - ref.mean())}
    row |= {
        "helped": helped.sum(), "hurt": hurt.sum(), "net": helped.sum() - hurt.sum(),
        "both ✓": ((ref == 1) & (ret == 1)).sum(), "both ✗": ((ref == 0) & (ret == 0)).sum(),
        "p": binomtest(helped.sum(), n_flips).pvalue if n_flips else 1.0,
    }
    if pointwise:
        # Which class the flips land on: D+ = novel (gold 1), D- = lower-novelty (gold 0).
        for cls, cls_name in [(1, "D+"), (0, "D-")]:
            row[f"helped {cls_name}"] = (helped & (g.gold == cls)).sum()
            row[f"hurt {cls_name}"] = (hurt & (g.gold == cls)).sum()
    else:
        # Whether the flips are real verdict changes or moves in and out of a tie.
        trans = g.transition[helped | hurt]
        row |= {t: (trans == t).sum() for t in TRANSITIONS[PAIRWISE_METRIC]}
        row["tie% ref→ret"] = f"{100 * g.ref_tie.mean():.0f}→{100 * g.ret_tie.mean():.0f}"
    return pd.Series(row)


def flip_table(mode):
    t = (PAIRED[PAIRED["mode"] == mode]
         .groupby(["track", "model"]).apply(flip_stats, include_groups=False)
         .reindex(pd.MultiIndex.from_product([MERGES, MODELS], names=["track", "model"])))
    fmt = {c: "{:+.1f}" for c in t.columns if c.startswith("Δ")}
    fmt |= {c: "{:.1f}" for c in t.columns if c.endswith("_ref")} | {"p": "{:.3f}"}
    caption = {
        "pointwise": "Pointwise · macro-F1 (paper's metric) · helped/hurt = instance flipped ✗→✓ / ✓→✗",
        "pairwise": f"Pairwise · helped/hurt on {PAIRWISE_NAMES[PAIRWISE_METRIC]}",
    }[mode]
    return t.style.format(fmt, precision=0).set_caption(caption)


display(flip_table("pointwise"))
display(flip_table("pairwise"))

## 2. Threshold shift (pointwise)

% of ideas called novel, per gold class. If D+ and D− move together, retrieval shifted the threshold rather than improving discrimination (`ΔJ` ≈ 0, J = TPR − FPR).

In [ ]:
def threshold_stats(g):
    row = {}
    for cls, name in [(1, "D+"), (0, "D-")]:
        ref, ret = (100 * pd.to_numeric(p[g.gold == cls]).mean() for p in (g.ref_pred, g.ret_pred))
        row |= {f"novel% {name} ref": ref, f"Δ {name}": ret - ref}
    row["J ref"] = row["novel% D+ ref"] - row["novel% D- ref"]
    row["ΔJ"] = row["Δ D+"] - row["Δ D-"]
    row["ΔF1"] = 100 * (macro_f1(g.gold, g.ret_pred) - macro_f1(g.gold, g.ref_pred))
    return pd.Series(row)


t = (PAIRED[PAIRED["mode"] == "pointwise"]
     .groupby(["track", "model"]).apply(threshold_stats, include_groups=False)
     .reindex(pd.MultiIndex.from_product([MERGES, MODELS], names=["track", "model"])))
display(t.style.format("{:.1f}").format("{:+.1f}", subset=[c for c in t.columns if c.startswith("Δ")])
        .set_caption("Pointwise · % of ideas called novel, per gold class"))

## 4. Case viewer

`cases(...)` lists instances retrieval helped / hurt, the ones most judges flip on first. `show(...)` puts one instance side by side for one judge: ideas + reviewer signals, the retrieved work (with publication dates), and every call's reasoning without / with retrieval. `show_all(...)` does the same for all judges at once: one line per judge with its per-call verdicts without → with retrieval, click it for the reasoning.
In pairwise reasoning, "Idea-0/1" is the presentation order of that call; each call's header says which is D+.

In [ ]:
CALL_RE = re.compile(
    r"^=+\n(?:Pointwise evaluation of instance \S+ \[call \d+\]"
    r"|Comparison between Idea (\S+) \[idea0\] and Idea (\S+) \[idea1\])\n=+\n", re.M)
RW_RE = re.compile(r"\[Start Related Work(?: for idea(\d))?\](.*?)\[End Related Work", re.S)
PAPER_RE = re.compile(r"\[metadata-(\d+)\]\n(.*?)\[abstract-\1\]\n(.*?)(?=\s*\[metadata-\d+\]|\s*$)", re.S)


@cache
def instances(track, mode):
    d = DIRS[track, mode, MODELS[0], "current"]
    return {str(k): v for k, v in yaml.safe_load((ROOT / _extract_instances_path(d)).read_text()).items()}


@cache
def calls(track, mode, model, cond, iid):
    """[(shown, reasoning, vote)]. Pairwise: shown = (idea id shown as Idea-0, as Idea-1), vote = idea id voted for."""
    sub = "pointwise" if mode == "pointwise" else "comparisons"
    f = next(DIRS[track, mode, model, cond].glob(f"run_*/{sub}/{iid}.txt"), None)
    if f is None:
        return []
    parts = CALL_RE.split(f.read_text())
    out = []
    for a, b, body in zip(parts[1::3], parts[2::3], parts[3::3]):
        reasoning, _, choice = body.partition("\nChoice:")
        m = re.search(r'"novelty":\s*(\d)', choice)
        shown = None if a is None else (a, b)
        vote = None if m is None else (shown[int(m[1])] if shown else int(m[1]))
        out.append((shown, reasoning.split("Thinking Process:", 1)[-1].strip(), vote))
    return out


@cache
def paper_dates(track, mode, model):
    """{retrieved paper title: publication date}, from the retrieval cache the run read (dates aren't in the prompt)."""
    with open(DIRS[track, mode, model, "retrieval"] / "debug_accuracy_report.txt.md") as f:
        path = ROOT / re.search(r'"retrieval_cache_file":\s*"([^"]+)"', f.read(50_000))[1]
    dates = {}
    # The human-only pointwise cache was later overwritten by the web-search face-off; the original is *_old.json.
    for p in [path, path.with_name(f"{path.stem}_old.json")]:
        if not p.exists():
            continue
        entries = json.loads(p.read_text()).values()
        if mode == "pairwise":
            entries = [e for per_idea in entries for e in per_idea.values()]
        dates |= {c["title"]: str(c.get("publication_date") or c.get("year") or "?")
                  for e in entries for c in e.get("candidates", [])}
    return dates


@cache
def related_work(track, mode, model):
    """{instance id: {idea id: [paper]}}, parsed from the prompts the retrieval run actually sent."""
    dates = paper_dates(track, mode, model)
    f = next(DIRS[track, mode, model, "retrieval"].glob("run_*/batch_requests_*.jsonl"))
    out = {}
    for line in f.open():
        req = json.loads(line)
        _, iid, *shown = req["custom_id"].split("__")  # ptw__<id>__<call> / cmp__<id>__<idea0>__<idea1>__<call>
        if iid in out:
            continue
        prompt = (req.get("body") or req["params"])["messages"][0]["content"]
        out[iid] = {}
        for slot, block in RW_RE.findall(prompt):
            papers = []
            for k, meta, abstract in PAPER_RE.findall(block):
                p = {"k": k, **dict(l.split(": ", 1) for l in meta.strip().splitlines() if ": " in l)}
                papers.append(p | {"abstract": abstract.strip(), "date": dates.get(p.get("title"), "?")})
            out[iid][iid if mode == "pointwise" else shown[int(slot)]] = papers
    return out


def cases(track, mode, kind="hurt", model=None):
    """Instances by outcome change, most judges changing the same way first.

    kind: "helped" / "hurt" (instance correctness), or a transition such as "tie->ok" (ok = ✓, x = ✗, tie).
    """
    p = PAIRED[(PAIRED.track == track) & (PAIRED["mode"] == mode)]
    if kind == "helped":
        p = p[p.ret_ok & ~p.ref_ok]
    elif kind == "hurt":
        p = p[p.ref_ok & ~p.ret_ok]
    else:
        alias = {"ok": "✓", "x": "✗"}
        p = p[p.transition == "→".join(alias.get(s.strip(), s.strip()) for s in re.split("->|→", kind))]
    by_id = p.groupby("id").model.agg(list)
    out = pd.DataFrame({"n_models": by_id.str.len(), "models": by_id.str.join(", ")})
    if model:
        out = out[by_id.map(lambda ms: model in ms)]
    return out.sort_values("n_models", ascending=False)


CSS = """<style>
.rv{font-size:13px;line-height:1.45}
.rv .cols{display:grid;grid-template-columns:1fr 1fr;gap:10px}
.rv .box{border:1px solid rgba(128,128,128,.35);border-radius:6px;padding:6px 10px;margin:6px 0}
.rv pre{white-space:pre-wrap;font-family:inherit;margin:4px 0}
.rv summary{cursor:pointer}
.rv .ok{color:#2e9e5b;font-weight:600} .rv .bad{color:#d9534f;font-weight:600} .rv .dim{opacity:.65}
</style>"""
esc = html.escape


def _details(summary, body, open_=False):
    return f"<details{' open' if open_ else ''}><summary>{summary}</summary>{body}</details>"


def _mark(ok):
    return "<span class=ok>✓</span>" if ok else "<span class=bad>✗</span>"


def _idea_box(label, text, meta):
    signals = [f"+ {s}" for s in meta.get("positive_signals") or []] + [f"− {s}" for s in meta.get("negative_signals") or []]
    body = f"<pre>{esc(text)}</pre>"
    if signals:
        body += "<div class=dim>reviewer signals:<ul>" + "".join(f"<li>{esc(s)}</li>" for s in signals) + "</ul></div>"
    head = f"<b>{label}</b> · {esc(meta.get('title', ''))} <span class=dim>(rating {meta.get('rating')})</span>"
    return f"<div class=box>{head}{_details('idea', body)}</div>"


def _papers_box(label, papers):
    items = "".join(_details(
        f"[{p['k']}] {esc(p.get('title', ''))} <span class=dim>· {esc(p['date'])}</span>",
        f"<div class=dim>query: {esc(p.get('retrieval_query', ''))}<br>why: {esc(p.get('why_relevant', ''))}</div>"
        f"<pre>{esc(p['abstract'])}</pre>") for p in papers)
    return f"<div class=box><b>{label}</b>{items or ' <span class=dim>none</span>'}</div>"


def _context(track, mode, iid):
    """Ideas + retrieved-work HTML, and how to score / label / order this instance's calls."""
    inst = instances(track, mode)[iid]
    rw = related_work(track, mode, MODELS[0]).get(iid, {})  # retrieval is precomputed: every judge sees the same papers
    if mode == "pointwise":
        gold = int(inst["label"] == "POSITIVE")
        verdict_name = {1: "novel", 0: "not novel"}
        ideas = _idea_box(f"D{'+' if gold else '−'} · gold: {verdict_name[gold]}", inst["idea"], inst["metadata"])
        papers = _papers_box("retrieved work", rw.get(iid, []))
        vote_ok = lambda v: v == gold
        call_head = lambda shown, v: verdict_name.get(v, "no vote")
        order = lambda c: 0
    else:
        pos = str(inst["expected_winners"][0])
        neg = next(str(k) for k in inst["ideas"] if str(k) != pos)
        role = {pos: "D+", neg: "D−"}
        ideas = "<div class=cols>" + "".join(
            _idea_box(f"{role[i]} · idea {i}", inst["ideas"][int(i)], inst["metadata"][int(i)]) for i in (pos, neg)) + "</div>"
        papers = "<div class=cols>" + "".join(
            _papers_box(f"retrieved for {role[i]}", rw.get(i, [])) for i in (pos, neg)) + "</div>"
        vote_ok = lambda v: v == pos
        call_head = lambda shown, v: f"Idea-0 = {role[shown[0]]}, Idea-1 = {role[shown[1]]} → {role.get(v, 'no vote')}"
        order = lambda c: c[0][0] != pos  # D+ shown first, then D− shown first
    return ideas, papers, vote_ok, call_head, order


def _row(track, mode, model, iid):
    return PAIRED[(PAIRED.track == track) & (PAIRED["mode"] == mode) & (PAIRED.model == model) & (PAIRED.id == iid)].iloc[0]


def _reasoning_cols(track, mode, model, iid, vote_ok, call_head, order):
    """Every call's reasoning, without | with retrieval."""
    row = _row(track, mode, model, iid)
    cols = []
    for cond, prefix in [("current", "ref"), ("retrieval", "ret")]:
        cs = sorted(calls(track, mode, model, cond, iid), key=order)
        n_ok = sum(vote_ok(v) for _, _, v in cs)
        head = (f"<b>{'reference' if cond == 'current' else '+ retrieval'}</b> {_mark(row[f'{prefix}_ok'])}"
                f"{' (tie)' if row[f'{prefix}_tie'] else ''} <span class=dim>· {n_ok}/{len(cs)} calls correct</span>")
        body = "".join(_details(f"call {i + 1} · {call_head(shown, v)} {_mark(vote_ok(v))}", f"<pre>{esc(r)}</pre>")
                       for i, (shown, r, v) in enumerate(cs))
        cols.append(f"<div class=box>{head}{body}</div>")
    return f"<div class=cols>{''.join(cols)}</div>"


def show(track, mode, model, iid):
    """One judge on one instance."""
    iid = str(iid)
    ideas, papers, vote_ok, call_head, order = _context(track, mode, iid)
    row = _row(track, mode, model, iid)
    outcome = {(False, True): "helped", (True, False): "hurt", (True, True): "both ✓", (False, False): "both ✗"}[
        (row.ref_ok, row.ret_ok)]
    title = f"<h3>{track} · {mode} · {model} · #{iid} — {row.transition} ({outcome})</h3>"
    cols = _reasoning_cols(track, mode, model, iid, vote_ok, call_head, order)
    display(HTML(CSS + f"<div class=rv>{title}{ideas}{papers}{cols}</div>"))


def show_all(track, mode, iid):
    """Every judge on one instance: per-call verdicts without → with retrieval; click a judge for its reasoning."""
    iid = str(iid)
    ideas, papers, vote_ok, call_head, order = _context(track, mode, iid)
    judges = ""
    for model in MODELS:
        marks = {cond: "".join(_mark(vote_ok(v)) for _, _, v in sorted(calls(track, mode, model, cond, iid), key=order))
                 for cond in ("current", "retrieval")}
        summary = (f"<b>{model}</b> · without {marks['current']} → with {marks['retrieval']} "
                   f"<span class=dim>({_row(track, mode, model, iid).transition})</span>")
        judges += _details(summary, _reasoning_cols(track, mode, model, iid, vote_ok, call_head, order))
    title = f"<h3>{track} · {mode} · #{iid}</h3>"
    display(HTML(CSS + f"<div class=rv>{title}{ideas}{papers}<div class=box><b>judges</b>{judges}</div></div>"))

In [ ]:
# KIND: "helped" / "hurt", or a transition: "x->ok", "tie->ok", "ok->x", "ok->tie", "x->tie", "tie->x" (ok = ✓, x = ✗).
TRACK, MODE, MODEL, KIND = "human+gen", "pointwise", "gpt-5.4", "ok->x"

C = cases(TRACK, MODE, KIND, model=MODEL)
display(C.head(10))
show(TRACK, MODE, MODEL, C.index[1])

In [ ]:
# All judges on one instance: pointwise instances with transition KIND for at least MIN_JUDGES judges.
TRACK, KIND, MIN_JUDGES = "human+gen", "ok->x", 3

C = cases(TRACK, "pointwise", KIND).query("n_models >= @MIN_JUDGES")
C = C.assign(gold=[instances(TRACK, "pointwise")[i]["label"] for i in C.index],
             title=[instances(TRACK, "pointwise")[i]["metadata"]["title"] for i in C.index])
display(C)
show_all(TRACK, "pointwise", C.index[11])

In [ ]:
# Review set: MODEL's ✓→✗ flips, N_PER_CLASS per gold class. Clean flips first (all reference calls correct),
# then those more judges share (less likely sampling noise), then the most decisive (fewest correct retrieval calls).
MODEL, TRACK, N_PER_CLASS = "gpt-5.4", "human+gen", 5

inst = instances(TRACK, "pointwise")
R = cases(TRACK, "pointwise", "ok->x", model=MODEL)
gold = {i: int(inst[i]["label"] == "POSITIVE") for i in R.index}
calls_ok = lambda cond: [sum(v == gold[i] for *_, v in calls(TRACK, "pointwise", MODEL, cond, i)) for i in R.index]
R = R.assign(gold=[{1: "D+", 0: "D-"}[gold[i]] for i in R.index], ref_calls_ok=calls_ok("current"),
             ret_calls_ok=calls_ok("retrieval"), title=[inst[i]["metadata"]["title"] for i in R.index],
             num=R.index.astype(int))
R = R.sort_values(["ref_calls_ok", "n_models", "ret_calls_ok", "num"], ascending=[False, False, True, True])
R = pd.concat([R[R.gold == c].head(N_PER_CLASS) for c in ("D+", "D-")]).drop(columns="num")
display(R)
for i in R.index:
    show(TRACK, "pointwise", MODEL, i)